# Explicit sklearn Logistic Plugin

This notebook keeps estimator code outside core Aegis and registers it explicitly before config validation. YAML only references `model.plugin_id`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Any, Mapping

def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'research').exists():
            return path
    raise RuntimeError('Run this notebook from inside the aegis-rd repository')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from research.aegis_research.config import resolve_experiment_config
from research.aegis_research.experiments import run_experiment
from research.aegis_research.model_contracts import (
    POSITIVE_CLASS_PROBABILITY,
    ModelDataset,
    ModelExecutionContext,
    ModelFitResult,
    ModelPluginDeclaration,
    ModelPluginDefinition,
    ModelPredictionResult,
)
from research.aegis_research.model_export import export_model_bundle
from research.aegis_research.model_registry import ModelRegistry


In [ ]:
class SklearnLogisticPlugin:
    def fit(self, dataset: ModelDataset, *, params: Mapping[str, Any], context: ModelExecutionContext) -> ModelFitResult:
        del context
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                max_iter=int(params.get('max_iter', 1000)),
                random_state=int(params.get('random_state', 42)),
            )),
        ])
        if dataset.target is None:
            raise ValueError('fit dataset target is required')
        model.fit(dataset.features, dataset.target.astype(int))
        classes = tuple(model.named_steps['classifier'].classes_)
        return ModelFitResult(
            state={'model': model},
            observed_classes=classes,
            class_probability_columns={class_label: f'class_{class_label}_probability' for class_label in classes},
            diagnostics={'estimator': 'sklearn.linear_model.LogisticRegression'},
            state_metadata={'state_format': 'pickle'},
        )

    def predict(self, state: Any, dataset: ModelDataset, *, params: Mapping[str, Any], context: ModelExecutionContext) -> ModelPredictionResult:
        del params, context
        model = state['model']
        classes = tuple(model.named_steps['classifier'].classes_)
        columns = {class_label: f'class_{class_label}_probability' for class_label in classes}
        return ModelPredictionResult(
            probabilities=pd.DataFrame(
                model.predict_proba(dataset.features),
                index=dataset.row_index,
                columns=[columns[class_label] for class_label in classes],
            ),
            observed_classes=classes,
            class_probability_columns=columns,
        )


In [ ]:
def validate_params(params: Mapping[str, Any]) -> Mapping[str, str]:
    issues = {}
    if 'max_iter' in params and (not isinstance(params['max_iter'], int) or params['max_iter'] <= 0):
        issues['max_iter'] = 'must be a positive integer'
    if 'random_state' in params and not isinstance(params['random_state'], int):
        issues['random_state'] = 'must be an integer'
    return issues

def build_registry():
    registry = ModelRegistry()
    registry.register(ModelPluginDefinition(
        declaration=ModelPluginDeclaration(
            id='examples.sklearn_logistic',
            version='1.0.0',
            prediction_outputs=(POSITIVE_CLASS_PROBABILITY,),
            state_schema_version='example_sklearn_logistic_state.v1',
            package_versions={'sklearn': sklearn.__version__},
        ),
        plugin=SklearnLogisticPlugin(),
        validate_params=validate_params,
    ))
    return registry.freeze()


In [ ]:
scratch = TemporaryDirectory(prefix='aegis-model-plugin-')
registry = build_registry()
experiment = {
    'schema_version': 2,
    'name': 'sklearn_logistic_plugin_example',
    'output_dir': scratch.name,
    'data': {
        'source': 'synthetic',
        'symbols': ['SYN'],
        'start': '2020-01-01',
        'timeframe': '1D',
        'rows': 240,
        'seed': 42,
    },
    'indicators': {
        'invalid_value_policy': 'drop_rows',
        'specs': [
            {'id': 'returns', 'params': {'window': [1, 5, 20]}, 'outputs': ['returns'], 'model_features': [{'output': 'returns'}]},
            {'id': 'ma', 'params': {'window': [10, 30], 'wtype': 'simple'}, 'outputs': ['ma'], 'model_features': [{'output': 'ma', 'transform': 'distance_to_close'}]},
            {'id': 'volatility', 'params': {'window': [20]}, 'outputs': ['volatility'], 'model_features': [{'output': 'volatility'}]},
            {'id': 'rsi', 'params': {'window': [14], 'wtype': 'wilder'}, 'outputs': ['rsi'], 'model_features': [{'output': 'rsi', 'transform': 'scale_0_1'}]},
        ],
    },
    'labels': {
        'generator': {'kind': 'fixlb', 'params': {'n': 5}},
        'target': {
            'role': 'supervised_target',
            'source_output': 'labels',
            'select': {'params': {'n': 5}},
            'transform': {'name': 'threshold_future_return', 'version': 1, 'params': {'threshold': 0.0}},
        },
    },
    'split': {'kind': 'purged_kfold', 'n_folds': 3, 'n_test_folds': 1, 'max_splits': 3},
    'model': {
        'plugin_id': 'examples.sklearn_logistic',
        'min_train_samples': 50,
        'params': {'max_iter': 1000, 'random_state': 42},
    },
    'signals': {'policy': 'long_only_hysteresis', 'long_entry_threshold': 0.55, 'long_exit_threshold': 0.50, 'execution_timing': 'next_open'},
    'portfolio': {'entry_budget': 1.0, 'direction': 'longonly'},
    'report': {'freq': '1D', 'year_freq': '252D', 'min_oos_sharpe': 0.5, 'max_oos_drawdown': 0.35, 'min_oos_trades': 1},
}
resolved = resolve_experiment_config(experiment, model_registry=registry)
result = run_experiment(resolved)
result['status']


In [ ]:
# Optional producer-side export for another prediction-only runtime project.
# The consuming project must register reviewed plugin code and validate this metadata before loading native state.
export_model_bundle(
    result['run_dir'],
    model_artifact_id='validation.split_0.model',
    output_dir=Path(result['run_dir']) / 'exports' / 'split_0_model',
)


In [ ]:
scratch.cleanup()
